# FT-00b C# : LoRA hyperparams from scratch — ablation rang × alpha
**Navigation** : [Index](README.md) | [<< Précédent](FT-00a-LoRA-from-scratch-CSharp.ipynb) | [Suivant >>](FT-00c-LoRA-SOTA-Comparison-Python.ipynb)

**Version anglaise** : [FT-00b — LoRA hyperparams (en)](FT-00b-LoRA-Hyperparams-from-scratch-Python_en.ipynb)

**Objectif** : mesurer firsthand comment le couple `(r, alpha)` — rang de la décomposition et *scaling* — borne la qualité d'adaptation d'un réseau gelé. Pas de `peft`, pas de `loralib` — la même `LoRALinear` que [FT-00a](FT-00a-LoRA-from-scratch-CSharp.ipynb), entraînée sur **6 valeurs de `r`** × **3 valeurs d'`alpha`/r** = 18 configurations, sur la même mini-tâche jouet (Fashion-MNIST inversé, classifieur gelé 512→10).

**Prérequis** : [FT-00a](FT-00a-LoRA-from-scratch-CSharp.ipynb) — `LoRALinear` y est construite avec TorchSharp.

**Durée** : ~15 min · **Niveau** : intermédiaire · **Matériel** : CPU suffit.

**Position dans la série** : FT-00a démontait le mécanisme ; ce notebook ferme la question du **réglage** — quand un rang trop petit sous-adapte, comment le facteur alpha modifie les résultats, et pourquoi le scaling `alpha/r` est ce qui rend le rang et la magnitude **découplables**.

**Au Programme** : 1. reprendre le mécanisme `LoRALinear` · 2. assembler la mini-tâche (Fashion-MNIST inversé, classifieur gelé) · 3. pré-entraîner le modèle sur le domaine inversé · 4. lancer l'ablation des 18 configurations · 5. lire la table exactitude × budget × scaling · 6. situer les trois baselines · 7. trois exemples guidés, puis trois exercices.

[Version Python](FT-00b-LoRA-Hyperparams-from-scratch-Python.ipynb)

Les archives Fashion-MNIST sont lues depuis `COURSIA_FASHION_MNIST`. Les indices PCG64 seedés sont stockés dans [FT-00b-fashion-indices.json](FT-00b-fashion-indices.json), dont l’empreinte est vérifiée avant usage. Les images traversent le réseau à chaque lot ; aucune caractéristique n’est mise en cache.

### Vérification de l'environnement

In [1]:
#r "nuget:TorchSharp-cpu, 0.107.0"
// Configurer le moteur CPU et la graine pour le banc complet.
using System;
using System.IO;
using System.IO.Compression;
using System.Buffers.Binary;
using System.Diagnostics;
using System.Linq;
using System.Collections.Generic;
using TorchSharp;
using TorchSharp.Modules;
using static TorchSharp.torch;
using Tensor=TorchSharp.torch.Tensor;
torch.set_num_threads(2); torch.manual_seed(42);
Console.WriteLine($"TorchSharp {typeof(torch).Assembly.GetName().Version}, libtorch {torch.__version__}, CPU, threads {torch.get_num_threads()}, graine 42");

Installed Packages TorchSharp-cpu, 0.107.0

Loading extensions from `/workspace/coursia-toolchains/nuget/skiasharp/2.88.6/interactive-extensions/dotnet/SkiaSharp.DotNet.Interactive.dll`

TorchSharp 0.107.0.0, libtorch 2.10.0.0, CPU, threads 2, graine 42


### Moteur effectif

Le replay affiche son moteur, ses threads et sa graine. Les temps et scores dépendent du CPU et de l’entraînement.

## 1. Réutiliser la `LoRALinear` de FT-00a, même mécanisme

Le mécanisme est le même : `B = 0` au pas 0, `W` gelé, scaling `alpha / r`. On reprend la **même classe** que FT-00a (cellule 6) sous le nom `LoRALinear` — même init, même forward, même `merged_weight` — pour qu'aucune divergence de sémantique ne se glisse dans l'ablation.

In [2]:
// La base est gelée ; seules les matrices de bas rang reçoivent des gradients.
public class LoRALinear:torch.nn.Module<Tensor,Tensor>{
 public readonly Linear Base; public readonly Parameter A,B; public readonly double Scaling;
 public LoRALinear(Linear b,int rank,double alpha):base("LoRALinear"){
  if(rank<=0)throw new ArgumentException("rang strictement positif");Base=b;foreach(var p in Base.parameters())p.requires_grad_(false);Scaling=alpha/rank;
  A=torch.nn.Parameter(torch.randn(new long[]{rank,b.weight.shape[1]})/Math.Sqrt(b.weight.shape[1]));
  B=torch.nn.Parameter(torch.zeros(new long[]{b.weight.shape[0],rank}));RegisterComponents();
 }
 public override Tensor forward(Tensor x)=>Base.forward(x)+Scaling*torch.matmul(torch.matmul(x,A.transpose(0,1)),B.transpose(0,1));
 public Tensor MergedWeight()=>Base.weight+Scaling*torch.matmul(B,A);
}
Console.WriteLine("LoRALinear : base gelée, A aléatoire, B nul, scaling alpha/r.");

LoRALinear : base gelée, A aléatoire, B nul, scaling alpha/r.


### Mécanisme LoRA

La base est gelée ; A est aléatoire et B nul. Le facteur est `alpha / r`.

La couche enveloppée conserve une matrice `W` et son biais, mais leurs paramètres sont gelés. Deux matrices entraînables, `A` de forme `r × 512` et `B` de forme `10 × r`, construisent la correction. La sortie additionne le résultat de la couche dense et `(alpha/r) × (x Aᵀ) Bᵀ`. Le calcul évite ainsi de former une matrice dense `B A` à chaque passe, même si cette matrice peut être construite pour l'analyse ou la fusion.

B est initialisée à zéro : avant tout apprentissage, la sortie de l'adaptateur est exactement celle de sa base. La cellule d'ablation vérifie cette identité sur de vraies caractéristiques. Le backward alloue des gradients aux facteurs entraînables ; les paramètres gelés ne sont pas remis à jour. Le rang et le scaling répondent à deux questions différentes : le premier règle la taille de la factorisation, le second multiplie son effet.

## 2. Mini-tâche : Fashion-MNIST inversé, classifieur gelé 512→10

On reprend la même tâche jouet que FT-00a (mini-tâche §3), avec un sous-ensemble pour rester sous 15 min CPU : 10 000 images d'entraînement, 2 000 de test. Le modèle jouet est un **petit CNN** pré-entraîné sur le domaine inversé ; on adapte **uniquement** sa tête `Linear(512, 10)` via LoRA.

Le CNN est gelé après pré-entraînement : ses 512 features servent de représentation fixe, et la tâche d'adaptation se réduit à une tête `512 → 10` — la configuration la plus favorable à LoRA (peu de paramètres, changement de domaine net).

In [3]:
// Lire le corpus et vérifier le hash et l’unicité des indices PCG64.
var dossier=Environment.GetEnvironmentVariable("COURSIA_FASHION_MNIST") ?? throw new Exception("Définir COURSIA_FASHION_MNIST vers les archives Fashion-MNIST");
int Entier(Stream s){Span<byte>b=stackalloc byte[4];s.ReadExactly(b);return BinaryPrimitives.ReadInt32BigEndian(b);}
(Tensor,Tensor) Charger(string prefix,int count){using var fi=File.OpenRead(Path.Combine(dossier,$"{prefix}-images-idx3-ubyte.gz"));using var im=new GZipStream(fi,CompressionMode.Decompress);using var fl=File.OpenRead(Path.Combine(dossier,$"{prefix}-labels-idx1-ubyte.gz"));using var la=new GZipStream(fl,CompressionMode.Decompress);if(Entier(im)!=2051||Entier(im)!=count||Entier(im)!=28||Entier(im)!=28||Entier(la)!=2049||Entier(la)!=count)throw new InvalidDataException("IDX inattendu");var pixels=new byte[count*28*28];var labels=new byte[count];im.ReadExactly(pixels);la.ReadExactly(labels);return(torch.tensor(pixels).to_type(ScalarType.Float32).div_(255).reshape(count,1,28,28),torch.tensor(labels).to_type(ScalarType.Int64));}
var (images,labels)=Charger("train",60000);var (testImages,testLabels)=Charger("t10k",10000);
var indexBytes=File.ReadAllBytes("FT-00b-fashion-indices.json");var hash=Convert.ToHexString(System.Security.Cryptography.SHA256.HashData(indexBytes)).ToLowerInvariant();if(hash!="9b61c00bfedbb90ae8ad600bb11c12b13ef05bb0d12efa781abbd3240e9ff130")throw new Exception("empreinte des indices inattendue");var doc=System.Text.Json.JsonDocument.Parse(indexBytes);
long[] trainArray=doc.RootElement.GetProperty("train").EnumerateArray().Select(x=>x.GetInt64()).ToArray();long[] testArray=doc.RootElement.GetProperty("test").EnumerateArray().Select(x=>x.GetInt64()).ToArray();
if(trainArray.Length!=10000||trainArray.Distinct().Count()!=10000||trainArray.Any(x=>x<0||x>=60000)||testArray.Length!=2000||testArray.Distinct().Count()!=2000||testArray.Any(x=>x<0||x>=10000))throw new Exception("indices invalides");
var trainIndex=torch.tensor(trainArray);var testIndex=torch.tensor(testArray);Console.WriteLine($"Fashion-MNIST : sous-ensemble {trainIndex.shape[0]} train et {testIndex.shape[0]} test, PCG64 graine 42, SHA-256 {hash}");

Fashion-MNIST : sous-ensemble 10000 train et 2000 test, PCG64 graine 42, SHA-256 9b61c00bfedbb90ae8ad600bb11c12b13ef05bb0d12efa781abbd3240e9ff130



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply r

### Sous-ensemble

Les indices PCG64 de la source sont fournis dans le JSON, contrôlé par SHA-256, sans doublon. Les images originales sont chargées une fois.

Le corpus original contient 60 000 images d'entraînement et 10 000 images de test. Le fichier d'indices conserve le tirage sans remise de `numpy.random.default_rng(42)` : 10 000 indices du premier ensemble puis 2 000 du second. La cellule contrôle l'empreinte SHA-256, les tailles, les bornes et l'absence de doublons. Le JSON ne contient aucune image et ne remplace pas les données Fashion-MNIST.

Les lots sont sélectionnés à partir du corpus chargé en mémoire et mélangés sans remise à chaque époque. Le réseau recalcule les convolutions et la couche dense pour chaque lot : aucun tableau de caractéristiques pré-calculées ne raccourcit le banc. Le modèle est pré-entraîné sur les négatifs `1-x`, puis l'adaptation est mesurée sur les images normales avec les mêmes étiquettes.

In [4]:
// Deux convolutions et une couche dense : aucun cache de caractéristiques entre lots.
public class MiniCNN:torch.nn.Module<Tensor,Tensor>{
 public readonly Conv2d c1,c2;public readonly Linear fc,head;
 public MiniCNN():base("MiniCNN"){c1=torch.nn.Conv2d(1,16,3,padding:1);c2=torch.nn.Conv2d(16,32,3,padding:1);fc=torch.nn.Linear(32*7*7,512);head=torch.nn.Linear(512,10);RegisterComponents();}
 public Tensor Features(Tensor x){x=torch.nn.functional.max_pool2d(torch.nn.functional.relu(c1.forward(x)),2);x=torch.nn.functional.max_pool2d(torch.nn.functional.relu(c2.forward(x)),2);return fc.forward(x.flatten(1));}
 public override Tensor forward(Tensor x)=>head.forward(Features(x));
}
double Eval(Func<Tensor,Tensor> predict,bool inverse=false){using var ng=torch.no_grad();long good=0;for(int i=0;i<2000;i+=512){using var scope=torch.NewDisposeScope();var idx=testIndex.narrow(0,i,Math.Min(512,2000-i));var x=testImages.index_select(0,idx);if(inverse)x=1-x;good+=predict(x).argmax(1).eq(testLabels.index_select(0,idx)).sum().item<long>();}return good/2000.0;}
void Epoch(Func<Tensor,Tensor> predict,torch.optim.Optimizer opt,bool inverse=false){using var perm=torch.randperm(10000);for(int i=0;i<10000;i+=256){using var scope=torch.NewDisposeScope();var positions=perm.narrow(0,i,Math.Min(256,10000-i));var idx=trainIndex.index_select(0,positions);var x=images.index_select(0,idx);if(inverse)x=1-x;var loss=torch.nn.functional.cross_entropy(predict(x),labels.index_select(0,idx));opt.zero_grad();loss.backward();opt.step();}}
Console.WriteLine("CNN à deux convolutions, couche dense 512 et tête 512→10 ; chaque lot traverse tout le réseau.");

CNN à deux convolutions, couche dense 512 et tête 512→10 ; chaque lot traverse tout le réseau.


### Architecture

Ce modèle comporte deux convolutions, une couche dense vers 512 et une tête. FT-00a emploie trois convolutions : les architectures sont distinctes.

La première convolution transforme une image `1 × 28 × 28` en 16 cartes, suivies d'une réduction par deux. La seconde produit 32 cartes et une nouvelle réduction donne `32 × 7 × 7`. Après aplatissement, une couche dense produit 512 caractéristiques et une tête linéaire fournit les dix logits. Cette couche dense n'est pas remplacée par la tête LoRA : elle reste dans le backbone.

FT-00a emploie au contraire trois blocs convolutifs et une tête directe de taille `576 → 10`. Le nom de famille et la tâche sont communs, pas l'architecture. Dans cette ablation, après pré-entraînement sur les négatifs, le backbone est gelé ; seules les matrices de l'adaptateur de tête reçoivent des mises à jour. Les logits et l'accuracy sont calculés sur des images jamais utilisées pour l'entraînement.

## 3. Pré-entraîner le modèle sur le domaine inversé

Comme dans FT-00a, deux époques d’Adam au taux `1e-3` utilisent les négatifs `1-x`, avant les évaluations sur négatif et sur images normales. Ici, le CNN est distinct : deux convolutions et une couche dense vers 512 caractéristiques. Les données aussi sont un sous-ensemble fixe de 10 000 images pour l’entraînement et 2 000 pour le test.

In [5]:
// Pré-entraîner le modèle complet sur les négatifs, puis mesurer la cible.
var baseModel=new MiniCNN();using(var opt=torch.optim.Adam(baseModel.parameters(),lr:1e-3)){for(int e=0;e<2;e++)Epoch(baseModel.forward,opt,true);}var accInv=Eval(baseModel.forward,true);var accNormal=Eval(baseModel.forward);Console.WriteLine($"base inversée {accInv:F4} ; cible normale {accNormal:F4}");foreach(var p in baseModel.parameters())p.requires_grad_(false);

base inversée 0.7955 ; cible normale 0.0215


### Base

Comparez l’exactitude du domaine inversé appris et celle de la cible normale dans la sortie précédente.

## 4. Ablation `r × alpha` : 18 configurations

On adapte la tête `head: Linear(512, 10)` via LoRA pour **18 combinaisons** :

- `r ∈ {1, 2, 4, 8, 16, 32}` — 6 valeurs de rang
- `alpha/r ∈ {0.5, 1.0, 2.0}` — 3 valeurs de scaling effectif

Le scaling effectif `alpha/r` est ce qui **découple** la magnitude du rang : avec `alpha/r = 1`, doubler `r` ne change pas l'amplitude du signal appris. Avec `alpha/r = 0.5`, le signal est atténué ; avec `alpha/r = 2`, il est amplifié.

Chaque configuration repart d'une tête neuve, à graine fixe : chaque construction repart de la même graine, mais les tenseurs ont des formes différentes selon le rang, et la colonne `cumul` date le coût de chaque run (le total est imprimé en dernière ligne). 18 runs de 2 epochs sur 10 000 images : c'est le prix de la mesure, contenu dans le budget annoncé en tête (~15 min sur CPU) ; les durées mesurées dépendent du CPU et de sa charge.

In [6]:
// Repartir d’une tête fraîche par configuration et entraîner deux époques complètes.
var ranks=new[]{1,2,4,8,16,32};var scalings=new[]{0.5,1.0,2.0};var results=new List<(int rank,double alpha,double scale,long count,double acc,double sec)>();var chrono=Stopwatch.StartNew();
LoRALinear FreshHead(int rank,double scale){torch.manual_seed(42);var h=torch.nn.Linear(512,10);using(var ng=torch.no_grad()){h.weight.uniform_(-1/Math.Sqrt(512),1/Math.Sqrt(512));h.bias.zero_();}return new LoRALinear(h,rank,rank*scale);}
// Vérifier l'identité initiale sur de vraies caractéristiques avant l'ablation.
using(var checkHead=FreshHead(4,1.0)) using(var ng=torch.no_grad()) using(var scope=torch.NewDisposeScope()){
 var features=baseModel.Features(images.index_select(0,trainIndex.narrow(0,0,32)));
 var err=(checkHead.forward(features)-checkHead.Base.forward(features)).abs().max().item<float>();
 Console.WriteLine($"Identité initiale LoRA : écart {err:E3}");
 if(err!=0)throw new Exception("identité initiale non vérifiée");
}
foreach(int rank in ranks)foreach(double scale in scalings){using var h=FreshHead(rank,scale);Func<Tensor,Tensor> predict=x=>h.forward(baseModel.Features(x));using var opt=torch.optim.Adam(h.parameters().Where(p=>p.requires_grad),lr:1e-3);for(int ep=0;ep<2;ep++)Epoch(predict,opt);double acc=Eval(predict);long count=h.parameters().Where(p=>p.requires_grad).Sum(p=>p.numel());if(count!=rank*522)throw new Exception("budget inattendu");results.Add((rank,rank*scale,scale,count,acc,chrono.Elapsed.TotalSeconds));Console.WriteLine($"r={rank}, alpha={rank*scale:F1}, alpha/r={scale:F1}, paramètres={count}, accuracy={acc:F4}, cumul={chrono.Elapsed.TotalSeconds:F1}s");}

Identité initiale LoRA : écart 0.000E+000


r=1, alpha=0.5, alpha/r=0.5, paramètres=522, accuracy=0.2180, cumul=5.4s


r=1, alpha=1.0, alpha/r=1.0, paramètres=522, accuracy=0.2280, cumul=10.3s


r=1, alpha=2.0, alpha/r=2.0, paramètres=522, accuracy=0.2225, cumul=13.6s


r=2, alpha=1.0, alpha/r=0.5, paramètres=1044, accuracy=0.2130, cumul=16.6s


r=2, alpha=2.0, alpha/r=1.0, paramètres=1044, accuracy=0.2125, cumul=19.7s


... (9 outputs left out)

r=16, alpha=32.0, alpha/r=2.0, paramètres=8352, accuracy=0.6070, cumul=56.1s


r=32, alpha=16.0, alpha/r=0.5, paramètres=16704, accuracy=0.5640, cumul=59.8s


r=32, alpha=32.0, alpha/r=1.0, paramètres=16704, accuracy=0.6215, cumul=63.4s


r=32, alpha=64.0, alpha/r=2.0, paramètres=16704, accuracy=0.6580, cumul=67.5s


### Ablation

Les 18 lignes proviennent chacune de deux époques et d’une tête LoRA fraîche. Les résultats dépendent du jeu, des poids et du régime CPU ; aucune tendance n’est garantie a priori.

Le tableau couvre le produit cartésien des six rangs `1, 2, 4, 8, 16, 32` et des trois rapports `alpha/r = 0,5 ; 1 ; 2`. Chaque cas crée une nouvelle tête avec la graine prescrite et exécute deux époques de 10 000 exemples. La forme des matrices change avec le rang : employer une même graine ne signifie pas que chaque paramètre corresponde élément par élément entre rangs. Les lignes sont comparables par protocole et budget, pas par égalité de tous les poids.

La tête de base de chaque adaptateur est fraîche, initialisée puis gelée, comme dans la source ; elle ne reprend pas la tête pré-entraînée. Le backbone, lui, conserve les poids appris sur négatifs. Cette distinction explique que l'ablation réponde à une question précise sur une représentation gelée et une nouvelle tête, tandis que le témoin full réentraîne la tête pré-entraînée. Les temps cumulés documentent ce run et ne sont pas des constantes de la méthode.

## 5. Table récapitulative : exactitude × budget × scaling

Deux vues, lues directement dans les 18 mesures de la section 4 : **exactitude** par `(r, alpha/r)` et **budget** (params entraînables) par `r`. La table ne porte pas de courbe de convergence — 2 epochs ne seraient de toute façon pas lisibles — c'est le rôle de l'exemple guidé 3 (trajectoire sur 6 epochs, section 7).

In [7]:
// Lire les 18 scores et les budgets à partir des résultats calculés.
Console.WriteLine("Exactitude par rang et alpha/r :");foreach(var r in ranks){Console.WriteLine($"r={r} : "+string.Join(" | ",scalings.Select(s=>results.Single(x=>x.rank==r&&x.scale==s).acc.ToString("F4"))));}foreach(var r in ranks)Console.WriteLine($"r={r} : {results.First(x=>x.rank==r).count} paramètres ; tête pleine 5130");

Exactitude par rang et alpha/r :


r=1 : 0.2180 | 0.2280 | 0.2225


r=2 : 0.2130 | 0.2125 | 0.2270


r=4 : 0.2730 | 0.3100 | 0.3430


r=8 : 0.3790 | 0.4235 | 0.5130


r=16 : 0.4670 | 0.5445 | 0.6070


... (3 outputs left out)

r=4 : 2088 paramètres ; tête pleine 5130


r=8 : 4176 paramètres ; tête pleine 5130


r=16 : 8352 paramètres ; tête pleine 5130


r=32 : 16704 paramètres ; tête pleine 5130


### Compromis

Chaque rang coûte `r × (512 + 10)` paramètres. Le tableau d’exactitude est à lire avec ces budgets.

Une tête LoRA de rang `r` utilise `r × 512 + 10 × r = 522r` paramètres entraînables. La tête dense avec biais compte `512 × 10 + 10 = 5 130` paramètres. À faible rang, le budget LoRA est plus petit ; lorsque le rang grandit, il peut dépasser celui de la tête pleine. Cette comparaison de nombres ne détermine pas à elle seule la qualité obtenue.

Pour lire le tableau, comparez d'abord les trois scalings à rang fixé, puis les rangs à scaling fixé. Une variation d'accuracy peut venir de la capacité, de l'amplitude du delta ou de l'optimisation sous le budget de deux époques. Une seule graine par configuration ne mesure pas la variabilité entre initialisations. Les valeurs affichées constituent les observations du replay, et non des seuils universels.

## 6. Comparaison aux baselines

Trois baselines à comparer aux 18 configurations :

- **Base gelée sur cible** : `acc_normal` du modèle de base (sous le hasard)
- **Full fine-tuning de la tête** : `Linear(512, 10)` entraînée sans contrainte, mêmes 2 epochs (étalon or, baseline **MED**)
- **LoRA r=4, alpha=4** (FT-00a) : la configuration de référence

Le full est l'**étalon or** : même tête, mêmes 2 epochs, aucune contrainte de rang — il fournit un témoin non contraint en rang, qui peut être dépassé sur ce jeu et ce budget. La base gelée, elle, n'apprend rien : c'est le plancher sous le hasard.

In [8]:
// Adapter toute la tête comme témoin à budget et données identiques.
foreach(var p in baseModel.head.parameters())p.requires_grad_(true);torch.manual_seed(42);using(var opt=torch.optim.Adam(baseModel.head.parameters(),lr:1e-3)){for(int e=0;e<2;e++)Epoch(baseModel.forward,opt);}var accFull=Eval(baseModel.forward);foreach(var p in baseModel.head.parameters())p.requires_grad_(false);Console.WriteLine($"base cible {accNormal:F4}, full tête {accFull:F4} (5130 paramètres), LoRA r=4 alpha/r=1 {results.Single(x=>x.rank==4&&x.scale==1).acc:F4}");

base cible 0.0215, full tête 0.4965 (5130 paramètres), LoRA r=4 alpha/r=1 0.3100


### Baselines

Comparez la base gelée, l’adaptation de toute la tête et LoRA à rang 4, sur le même sous-ensemble.

La base évalue la tête pré-entraînée sans adaptation sur la cible. Le témoin full dégèle cette tête et l'entraîne pendant deux époques, sans modifier le backbone. LoRA, pour sa part, entraîne seulement ses deux facteurs autour d'une tête fraîche gelée. Les trois lignes partagent le sous-ensemble et la métrique, mais la tête de départ du témoin full diffère de celle des 18 essais LoRA.

Un modèle plus contraint peut dépasser le témoin full sur un jeu et un budget donnés : régularisation implicite, initialisation et dynamique d'optimisation influencent le résultat. Il faut donc lire le témoin comme un point de comparaison, pas comme une borne mathématique. Les 5 130 paramètres du témoin incluent son biais ; les comptes LoRA affichés correspondent uniquement aux facteurs A et B.

## 7. Exemples guidés et exercices

Trois exemples guidés, suivis de trois exercices de portée distincte. Chaque exemple démontre une technique ; chaque exercice demande à l'étudiant de mesurer une grandeur **différente** de celle de l'exemple correspondant. Les corrigés des exemples sont **dans** les exemples (ils s'exécutent et impriment leurs sorties réelles), pas dans une section voisine.

**Méthode** : lire les trois exemples guidés et leurs sorties, puis tenter les trois exercices (stubs C.1 conformes — `null` et commentaires TODO).

Chaque exercice réutilise **à l'identique** le code de l'exemple qui le précède — seule la grandeur mesurée change : aucun ingrédient nouveau n'est requis pour les trois.

### Exemple guide 1 : rapport params/accuracy pour `alpha/r = 1.0`

Pour `alpha/r = 1.0`, calculer le **rapport `params / accuracy`** pour chaque `r ∈ {1, 2, 4, 8, 16, 32}` à partir des 18 mesures de la section 4. Identifier le `r` qui **minimise** ce rapport — c'est le rendement marginal du rang sur cette mini-tâche.

**Pourquoi c'est utile** : un rang `r=1` peut sembler efficient (peu de params) mais sous-adapter ; un rang `r=32` peut sembler précis mais coûteux en mémoire d'inférence. Le rapport `params / accuracy` estime le **coût par point d'exactitude** — c'est ce qu'un ingénieur doit regarder pour choisir le rang à déployer.

### Exemple guidé 2 : comparer les normes pour deux scalings

Pour `r=4`, entraîner séparément les configurations `alpha/r=1` et `2`. Mesurer la norme du delta effectif ainsi que celles des facteurs A et B. Le scaling modifie à la fois le résultat de la multiplication et la trajectoire d'optimisation ; il ne suffit pas de supposer un doublement de la norme finale.

### Exemple guidé 3 : trajectoire sur six époques

Pour `r=4` et `alpha/r=2`, entraîner six époques et mesurer l'exactitude après chacune. Observer si la trajectoire monte, plafonne ou baisse sur ce replay, puis discuter ce qu'un arrêt anticipé pourrait faire. Le test ne garantit pas que cette tendance se reproduise à une autre graine.

### Exercice 1 : `alpha/r = 0.5` — quand le rang ne suffit plus


Refaire l'analyse de l'exemple guide 1 (rapport `params / accuracy`) pour **`alpha/r = 0.5`** sur les 6 rangs. Comparer au régime `alpha/r = 1.0` : le rapport `params / accuracy` augmente-t-il plus vite sur le scaling bas ? Identifier le rang où le **rendement marginal** du rang commence à décliner — discutez les limites de ce rapport.

**Indice 1** : reprendre la boucle la boucle sur `ranks` de l'exemple guide 1, en remplaçant `scale == 1.0` par `scale == 0.5`.

**Indice 2** : pour comparer, calculer le ratio `params/acc(0.5) / params/acc(1.0)` pour chaque rang — un ratio constant indiquerait que le scaling est un simple facteur multiplicatif.

### Exercice 2 : norme de `B A` selon le scaling

Pour les trois scalings `0.5`, `1.0` et `2.0`, à rang 4, réentraîner les adaptateurs et comparer la norme du produit non mis à l'échelle `B A`. `TrainMeasure` retourne la norme du delta effectif qui inclut le scaling ; déduisez-en la grandeur à comparer. Discutez les résultats sans présumer d'une tendance.

### Exercice 3 : trajectoire et arrêt anticipé

Entraîner `r=8`, `alpha/r=1` pendant trois époques et mesurer l'accuracy à chaque étape. Comparer la valeur finale à celle de la même configuration à deux époques dans `results`. La trajectoire à rang 4 sert de patron, mais ne garantit pas le comportement du rang 8.

Pour une décision d'arrêt en production, distinguer jeu de validation et jeu de test final.

In [9]:
// Mesurer le coût en paramètres par unité d’exactitude.
var ratios=results.Where(x=>x.scale==1).Select(x=>(x.rank,x.count,x.acc,ratio:x.count/Math.Max(x.acc,1e-6))).ToArray();foreach(var x in ratios)Console.WriteLine($"r={x.rank}, paramètres={x.count}, accuracy={x.acc:F4}, paramètres/accuracy={x.ratio:F1}");var best=ratios.OrderBy(x=>x.ratio).First();Console.WriteLine($"minimum : r={best.rank}, ratio {best.ratio:F1}");

r=1, paramètres=522, accuracy=0.2280, paramètres/accuracy=2289.5


r=2, paramètres=1044, accuracy=0.2125, paramètres/accuracy=4912.9


r=4, paramètres=2088, accuracy=0.3100, paramètres/accuracy=6735.5


r=8, paramètres=4176, accuracy=0.4235, paramètres/accuracy=9860.7


r=16, paramètres=8352, accuracy=0.5445, paramètres/accuracy=15338.8


r=32, paramètres=16704, accuracy=0.6215, paramètres/accuracy=26876.9


minimum : r=1, ratio 2289.5


### Rendement

Le rapport paramètres/exactitude donne un coût par unité de score pour ce banc ; il ne résume pas tous les critères de choix.

In [10]:
// Réentraîner les deux scalings pour mesurer les normes du delta et des facteurs.
(double,double,double) TrainMeasure(double scale){using var h=FreshHead(4,scale);Func<Tensor,Tensor> pred=x=>h.forward(baseModel.Features(x));using var opt=torch.optim.Adam(h.parameters().Where(p=>p.requires_grad),lr:1e-3);for(int e=0;e<2;e++)Epoch(pred,opt);using var ng=torch.no_grad();return ((h.Scaling*torch.matmul(h.B,h.A)).norm().item<float>(),h.A.norm().item<float>(),h.B.norm().item<float>());}
foreach(double s in new[]{1.0,2.0}){var n=TrainMeasure(s);Console.WriteLine($"alpha/r={s:F1}, ||delta W||={n.Item1:F4}, ||A||={n.Item2:F4}, ||B||={n.Item3:F4}");}

alpha/r=1.0, ||delta W||=1.1616, ||A||=3.8783, ||B||=0.4392


alpha/r=2.0, ||delta W||=1.7508, ||A||=3.5818, ||B||=0.3882


### Normes

La norme affichée est celle du delta effectif, qui inclut le scaling. Comparer aussi les normes A et B.

Le produit `B A` est la correction avant mise à l'échelle ; `delta W = (alpha/r) B A` est la modification effective ajoutée à W. L'exemple entraîne séparément deux têtes de rang 4 et imprime la norme du delta effectif ainsi que celles de A et de B. Comparer simplement les normes de B ne suffit pas, car A et B peuvent changer simultanément.

Multiplier le scaling ne multiplie pas nécessairement le delta final par le même facteur : modifier le scaling change aussi les gradients et donc la trajectoire d'apprentissage. Pour examiner le produit non mis à l'échelle, l'exercice propose de diviser la norme du delta par le scaling correspondant. Cette mesure reste dépendante des poids, des données et des deux époques utilisées.

In [11]:
// Mesurer l’exactitude après chacune des six époques.
using(var h=FreshHead(4,2.0)){Func<Tensor,Tensor> pred=x=>h.forward(baseModel.Features(x));using var opt=torch.optim.Adam(h.parameters().Where(p=>p.requires_grad),lr:1e-3);for(int e=1;e<=6;e++){Epoch(pred,opt);Console.WriteLine($"époque {e} : accuracy {Eval(pred):F4}");}}

époque 1 : accuracy 0.2680


époque 2 : accuracy 0.3430


époque 3 : accuracy 0.5200


époque 4 : accuracy 0.5790


époque 5 : accuracy 0.5995


époque 6 : accuracy 0.6115


### Trajectoire

Les six valeurs retracent les évaluations après chaque époque. Une trajectoire locale ne prouve pas un comportement universel.

L'exemple suit une tête de rang 4 avec `alpha/r = 2` pendant six époques. Chaque valeur est une accuracy sur les mêmes 2 000 images de test, après une nouvelle époque d'entraînement ; elle ne résulte pas d'une interpolation entre deux mesures. La courbe peut monter, plafonner ou se dégrader selon le run. Le dernier score n'est pas nécessairement le meilleur, et ces six observations ne suffisent pas à fixer un seuil d'arrêt pour d'autres modèles.

L'exercice suivant invite à changer de configuration et à comparer une trajectoire plus courte avec le tableau à deux époques. Un véritable early stopping réserverait idéalement un jeu de validation distinct du test final, afin de ne pas sélectionner l'époque sur les mêmes données que celles utilisées pour annoncer la performance.

In [12]:
// TODO étudiant : construire le ratio paramètres / accuracy pour alpha/r=0.5 et comparer à ratios.
List<(int,double)> ratiosLow=null;
Console.WriteLine("Exercice 1 à compléter : construire et comparer ratiosLow");

Exercice 1 à compléter : construire et comparer ratiosLow


La cellule suivante poursuit le contrôle ou l’exercice précédent.

In [13]:
// TODO étudiant : reprendre TrainMeasure pour 0.5, 1.0, 2.0 et normaliser la norme du delta par le scaling.
object resultNorm=null;
Console.WriteLine("Exercice 2 à compléter : norme normalisée");

Exercice 2 à compléter : norme normalisée


La cellule suivante poursuit le contrôle ou l’exercice précédent.

In [14]:
// TODO étudiant : entraîner r=8, alpha/r=1 pendant trois époques et comparer à results.
object resultEarly=null;
Console.WriteLine("Exercice 3 à compléter : trajectoire et comparaison");

Exercice 3 à compléter : trajectoire et comparaison


## Résumé

Le rang fixe le budget, tandis que alpha/r règle le facteur appliqué au delta. L’ablation, les baselines et les normes ci-dessus permettent de confronter ces choix sur le même jeu.